# Heart Disease UCI - EDA, Training & Inference
Interactive companion to the scripts in `heart_disease/`. Select the `.venv` kernel in VS Code (install `ipykernel` if prompted).

In [ ]:
import os, sys
from pathlib import Path
# make the project root importable when the notebook runs from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
%matplotlib inline
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns

## 1. Data acquisition & cleaning

In [ ]:
from heart_disease.download_data import download
from heart_disease.data import load_raw_data, clean_data
raw = load_raw_data(download())
print(raw.shape)
raw.isna().sum()

In [ ]:
df = clean_data(raw)
df.head()

In [ ]:
df.describe().T

## 2. EDA

In [ ]:
from heart_disease.config import NUMERIC_FEATURES, TARGET
sns.countplot(x=TARGET, data=df); plt.title('Class balance'); plt.show()
df[NUMERIC_FEATURES].hist(bins=20, figsize=(12, 6)); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(11, 9))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.show()

## 3. Training (logs to MLflow)
Runs the full grid search for all model families (~1 min). View results with `mlflow ui --backend-store-uri sqlite:///mlflow.db`.

In [ ]:
from heart_disease.train import main as train_main
metadata = train_main(quick=False)
metadata['model_name'], metadata['metrics']

In [ ]:
pd.read_csv('reports/model_comparison.csv')

## 4. Inference

In [ ]:
import json
from heart_disease.predict import load_model, predict
sample = json.load(open('samples/sample_request.json'))
predict(load_model(), sample)